# Phase II — skeleton + graph healing

Input: `outputs/masks_deeplab/{id}_pred.png` from Phase I  
Output: `graphs_deeplab/{id}_graph.json`

CPU is enough. No training here — just geometry + NetworkX.

Masks should already exist on Drive from the DeepLab notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "scikit-image",
    "networkx",
    "opencv-python",
    "matplotlib",
    "tqdm",
])
print("packages ok")


In [ ]:
import os

DRIVE_BASE = "/content/drive/MyDrive/RouteResilience"
DATA_DIR   = f"{DRIVE_BASE}/datasets/train"
MASK_DIR   = f"{DRIVE_BASE}/outputs/masks_deeplab"
GRAPH_DIR  = f"{DRIVE_BASE}/outputs/graphs_deeplab"
VIS_DIR    = f"{DRIVE_BASE}/outputs/visualizations_deeplab"

for d in [GRAPH_DIR, VIS_DIR]:
    os.makedirs(d, exist_ok=True)

# Phase II hyperparameters
MAX_BRIDGE_DIST  = 40
MAX_ANGLE_DIFF   = 35
MIN_ROAD_AREA    = 80
MIN_EDGE_LENGTH  = 10

# Demo samples — only processed if mask file exists
PROCESS_IDS = ["493626", "477671", "422265", "194764"]

print(f"Mask dir  : {MASK_DIR}")
print(f"Graph dir : {GRAPH_DIR}")
print(f"Vis dir   : {VIS_DIR}")
print(f"Samples   : {PROCESS_IDS}")


In [ ]:
import math
import json
import numpy as np
import cv2
import networkx as nx
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize, remove_small_objects


def preprocess_mask(mask, min_area=MIN_ROAD_AREA):
    """Close small gaps and remove noise blobs."""
    mask = (mask > 127).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask_bool = remove_small_objects(mask.astype(bool), min_size=min_area)
    return mask_bool.astype(np.uint8)


def extract_skeleton(mask):
    clean = preprocess_mask(mask)
    skel = skeletonize(clean > 0)
    return skel.astype(np.uint8), clean


def neighbors8(r, c, h, w):
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0:
                continue
            nr, nc = r + dr, c + dc
            if 0 <= nr < h and 0 <= nc < w:
                yield nr, nc


def build_graph(skel):
    """
    Extract nodes at intersections (degree >= 3) and endpoints (degree == 1).
    Trace edges as polylines between nodes along degree-2 pixels.
    """
    h, w = skel.shape
    skel_set = set(zip(*np.where(skel > 0)))

    degree = {}
    for r, c in skel_set:
        degree[(r, c)] = sum(
            1 for nr, nc in neighbors8(r, c, h, w) if (nr, nc) in skel_set
        )

    node_pixels = {p for p, d in degree.items() if d != 2}
    G = nx.Graph()
    for i, (r, c) in enumerate(node_pixels):
        G.add_node(i, y=int(r), x=int(c), pixel=(r, c))

    pixel_to_node = {G.nodes[n]["pixel"]: n for n in G.nodes}
    visited = set()

    for start_node in G.nodes:
        sr, sc = G.nodes[start_node]["pixel"]
        for nr, nc in neighbors8(sr, sc, h, w):
            if (nr, nc) not in skel_set:
                continue
            key = tuple(sorted([(sr, sc), (nr, nc)]))
            if key in visited:
                continue

            path = [(sr, sc), (nr, nc)]
            pr, pc = sr, sc
            cr, cc = nr, nc

            while (cr, cc) not in node_pixels:
                nbrs = [
                    p for p in neighbors8(cr, cc, h, w)
                    if p in skel_set and p != (pr, pc)
                ]
                if not nbrs:
                    break
                pr, pc = cr, cc
                cr, cc = nbrs[0]
                path.append((cr, cc))

            if (cr, cc) in node_pixels:
                end_node = pixel_to_node[(cr, cc)]
                if start_node != end_node and len(path) >= MIN_EDGE_LENGTH:
                    G.add_edge(
                        start_node, end_node,
                        pts=path, length=float(len(path)), healed=False,
                    )
                    visited.add(key)

    return G


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True


def endpoint_direction(G, node):
    """Unit vector along the road leaving an endpoint."""
    pixel = G.nodes[node]["pixel"]
    nbrs = [G.nodes[n]["pixel"] for n in G.neighbors(node)]
    if not nbrs:
        return (0.0, 0.0)
    dr = nbrs[0][0] - pixel[0]
    dc = nbrs[0][1] - pixel[1]
    mag = math.hypot(dr, dc) or 1.0
    return (dr / mag, dc / mag)


def angle_between(v1, v2):
    dot = max(-1.0, min(1.0, v1[0] * v2[0] + v1[1] * v2[1]))
    return math.degrees(math.acos(dot))


def heal_graph(G, max_dist=MAX_BRIDGE_DIST, max_angle=MAX_ANGLE_DIFF):
    """
    Bridge nearby aligned endpoints from different components.
    Red edges in visualization = healed bridges.
    """
    endpoints = [n for n in G.nodes if G.degree(n) == 1]
    if len(endpoints) < 2:
        return G, []

    uf = UnionFind(max(G.nodes) + 1 if G.nodes else 1)
    for u, v in G.edges:
        uf.union(u, v)

    candidates = []
    for i, n1 in enumerate(endpoints):
        p1 = G.nodes[n1]["pixel"]
        d1 = endpoint_direction(G, n1)
        for n2 in endpoints[i + 1:]:
            if uf.find(n1) == uf.find(n2):
                continue
            p2 = G.nodes[n2]["pixel"]
            dist = math.hypot(p1[0] - p2[0], p1[1] - p2[1])
            if dist > max_dist:
                continue
            d2 = endpoint_direction(G, n2)
            ang = angle_between(d1, (-d2[0], -d2[1]))
            if ang > max_angle:
                continue
            weight = dist + ang * 0.5
            candidates.append((weight, n1, n2, p1, p2))

    candidates.sort()
    bridges = []
    for weight, n1, n2, p1, p2 in candidates:
        if uf.union(n1, n2):
            G.add_edge(n1, n2, pts=[p1, p2], length=weight, healed=True)
            bridges.append({"source": n1, "target": n2, "weight": weight})

    return G, bridges


def conn_ratio(G):
    """Share of nodes in the largest component."""
    if G.number_of_nodes() == 0:
        return 0.0
    return len(max(nx.connected_components(G), key=len)) / G.number_of_nodes()





In [ ]:
def export_json(G, image_id, ratio_before, ratio_after, bridges, out_dir=GRAPH_DIR):
    """Write graph JSON for Phase III / dashboard."""
    nodes = [
        {
            "id": f"n{n}",
            "x": G.nodes[n]["x"],
            "y": G.nodes[n]["y"],
            "degree": G.degree(n),
        }
        for n in G.nodes
    ]
    edges = [
        {
            "id": f"e{u}_{v}",
            "source": f"n{u}",
            "target": f"n{v}",
            "weight": data.get("length", 1.0),
            "healed": data.get("healed", False),
            "coords": [[int(p[0]), int(p[1])] for p in data["pts"]],
        }
        for u, v, data in G.edges(data=True)
    ]
    out = {
        "image_id": image_id,
        "nodes": nodes,
        "edges": edges,
        "stats": {
            "num_nodes": len(nodes),
            "num_edges": len(edges),
            "connectivity_before": round(ratio_before, 4),
            "connectivity_after": round(ratio_after, 4),
            "bridges_added": len(bridges),
        },
    }
    path = f"{out_dir}/{image_id}_graph.json"
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    return out


def visualize(sat_img, mask, skel, G, image_id):
    """Save 4-panel PNG."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(sat_img)
    axes[0].set_title("Satellite Image")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("DeepLabV3+ Mask")

    axes[2].imshow(skel, cmap="gray")
    axes[2].set_title("Skeleton")

    axes[3].imshow(sat_img)
    for u, v, data in G.edges(data=True):
        pts = data["pts"]
        ys = [p[1] for p in pts]
        xs = [p[0] for p in pts]
        color = "red" if data.get("healed") else "lime"
        lw = 2.5 if data.get("healed") else 1.5
        axes[3].plot(xs, ys, color=color, linewidth=lw)
    for n in G.nodes:
        axes[3].plot(
            G.nodes[n]["x"], G.nodes[n]["y"],
            "o", color="yellow", markersize=4,
        )
    axes[3].set_title("Healed Graph (green=road, red=bridge)")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(f"Phase II DeepLab — {image_id}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    save_path = f"{VIS_DIR}/{image_id}_phase2.png"
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


def load_mask_for_sat(mask_path, sat_shape):
    """Load mask; resize to satellite image size with INTER_NEAREST if needed."""
    mask = cv2.imread(mask_path, 0)
    if mask is None:
        return None
    h, w = sat_shape[:2]
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


# run
results = []

for img_id in PROCESS_IDS:
    mask_path = f"{MASK_DIR}/{img_id}_pred.png"
    sat_path = f"{DATA_DIR}/{img_id}_sat.jpg"

    if not os.path.exists(mask_path):
        print(f"SKIP {img_id} — mask not found at {mask_path}")
        continue
    if not os.path.exists(sat_path):
        print(f"SKIP {img_id} — satellite image not found")
        continue

    sat_img = cv2.cvtColor(cv2.imread(sat_path), cv2.COLOR_BGR2RGB)
    mask = load_mask_for_sat(mask_path, sat_img.shape)
    if mask is None:
        print(f"SKIP {img_id} — could not read mask")
        continue

    skel, clean = extract_skeleton(mask)
    G = build_graph(skel)
    ratio_before = conn_ratio(G)
    G, bridges = heal_graph(G)
    ratio_after = conn_ratio(G)

    export_json(G, img_id, ratio_before, ratio_after, bridges)
    visualize(sat_img, clean, skel, G, img_id)

    results.append({
        "id": img_id,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "bridges": len(bridges),
        "connectivity_before": round(ratio_before, 3),
        "connectivity_after": round(ratio_after, 3),
    })
    print(
        f"[{img_id}] nodes={G.number_of_nodes()} edges={G.number_of_edges()} "
        f"bridges={len(bridges)} connectivity: {ratio_before:.2f} → {ratio_after:.2f}"
    )

print("\nresults:")
print(f"{'ID':<12} {'Nodes':>6} {'Edges':>6} {'Bridges':>8} {'Conn Before':>12} {'Conn After':>11}")
print("-" * 60)
for r in results:
    print(
        f"{r['id']:<12} {r['nodes']:>6} {r['edges']:>6} {r['bridges']:>8} "
        f"{r['connectivity_before']:>12.3f} {r['connectivity_after']:>11.3f}"
    )

if results:
    avg_before = np.mean([r["connectivity_before"] for r in results])
    avg_after = np.mean([r["connectivity_after"] for r in results])
    print(f"\nAvg connectivity: {avg_before:.3f} → {avg_after:.3f}")

print(f"\nGraphs saved to: {GRAPH_DIR}")
print(f"Visualizations saved to: {VIS_DIR}")


## Outputs

After a full run you should see:

- `outputs/graphs_deeplab/{id}_graph.json` — nodes, edges, coords, healed flag
- `outputs/visualizations_deeplab/{id}_phase2.png` — 4-panel figure for slides

Phase III reads the JSON files. If bridges look wrong, tweak `MAX_BRIDGE_DIST` in the config cell (40 px default).
